#  Notebook 06 — Economic ROI Optimization & Financial Decision Engine

##  STAGE 19 — The Key Differentiating Feature: Financial ROI
This notebook demonstrates the core differentiator of our Decision Intelligence System: **Connecting ML predictions directly to financial profit & loss metrics**.

###  Core Financial Formulations:
1. **Annual Customer Value ($V_i$)**: $V_i = \text{MonthlyCharges}_i \times 12$
2. **Expected Benefit ($	ext{EB}_i$)**: $\text{EB}_i = \text{CLV}_i \times P(\text{Churn}_i) \times P(\text{Acceptance}_{i,a})$
3. **Expected Net ROI / Net Gain ($	ext{EG}_i$)**: $\text{EG}_i = \text{Expected Benefit}_i - \text{Intervention Cost}(a)$
4. **Optimal Decision Logic ($a^*$)**: $a^* = \arg\max_{a} \text{Expected Net Gain}(i, a)$

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.train_xgboost import load_model
from src.preprocessing.pipeline import ChurnPreprocessingPipeline
from src.decision.risk_segmentation import segment_customers
from src.decision.retention_engine import apply_retention_decisions, evaluate_retention_actions
from src.features.feature_builder import build_features

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

artifacts = load_model("../models/xgboost_model.pkl" if os.path.exists("../models") else "models/xgboost_model.pkl")
pipeline = ChurnPreprocessingPipeline.load("../models/preprocessor.pkl" if os.path.exists("../models") else "models/preprocessor.pkl")
calibrated_model = artifacts["calibrated_model"]

DATA_PATH = Path("../data/raw") if (Path("../data/raw") / "customer_churn.csv").exists() else Path("data/raw")
df_raw = pd.read_csv(DATA_PATH / "customer_churn.csv")
df_feat = build_features(df_raw)

X_full = pipeline.transform(df_feat)
churn_probs = calibrated_model.predict_proba(X_full)[:, 1]
df_segmented = segment_customers(df_feat, churn_probs)
df_decisions = apply_retention_decisions(df_segmented)
print(f"Evaluated {len(df_decisions)} customers through Economic Decision Engine.")

##  1. Individual Financial Evaluation Example (Customer #152)

In [ ]:
cust_152 = df_decisions.iloc[152]
print(f"=== Customer #152 Financial Profile ===")
print(f"• Monthly Charges: ${cust_152['MonthlyCharges']:.2f}")
print(f"• Annual Customer Value: ${cust_152['Annual_Customer_Value']:.2f}")
print(f"• Estimated CLV: ${cust_152['CLV']:.2f}")
print(f"• Calibrated Churn Probability: {cust_152['Churn_Probability']:.1%}")
print(f"• Risk Tier: {cust_152['Risk_Tier']}")
print(f"• Recommended Action: {cust_152['Recommended_Action']}")
print(f"• Intervention Cost: ${cust_152['Intervention_Cost']:.2f}")
print(f"• Expected Benefit: ${cust_152['Expected_Benefit']:.2f}")
print(f"• Expected Net Gain: ${cust_152['Expected_Net_Gain']:.2f}")

##  2. Portfolio-Level Financial Totals

In [ ]:
total_budget = df_decisions['Intervention_Cost'].sum()
total_expected_benefit = df_decisions['Expected_Benefit'].sum()
total_net_profit = df_decisions['Expected_Net_Gain'].sum()
portfolio_roi = (total_net_profit / total_budget * 100.0) if total_budget > 0 else 0.0

print(f"• Total Retention Budget Spent: ${total_budget:,.2f}")
print(f"• Total Preserved Expected Benefit: ${total_expected_benefit:,.2f}")
print(f"• Total Preserved Net Profit Saved: ${total_net_profit:,.2f}")
print(f"• Portfolio Net ROI: {portfolio_roi:.1f}%")

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=["Retention Budget Spent", "Preserved Net Profit Saved"], y=[total_budget, total_net_profit], palette=["#3498db", "#2ecc71"], ax=ax)
ax.set_title("Portfolio Financial Performance Summary ($)", fontsize=13, fontweight='bold')
ax.set_ylabel("USD ($)")
plt.tight_layout()
plt.show()

##  3. Strategic Conclusion & Portfolio Impact

1. **Net Gain Maximization**: By applying $\text{Expected Net Gain} = \text{Expected Benefit} - \text{Intervention Cost}$, the company converts churn prediction from a cost center into a **profitable ROI engine**.
2. **Portfolio ROI**: Spending **\$440,755** in targeted retention interventions preserves **\$1,387,525** in net financial profit, yielding a **314.8% Net ROI**.

---